# Belebele k=8 Positional Bias — Clean, Non-Overlapping Confirmation

```
WHY THIS SCRIPT EXISTS
-----------------------
Earlier runs of this study reused a fixed data-shuffle seed across every
experiment stage. Because Belebele's train/test split ranges were taken
as prefixes of each other (e.g. a 60-item test set and a later 150-item
test set both started at the same row), and because the model uses
fully deterministic decoding (do_sample=False), many "new" predictions
in later runs were actually exact re-derivations of predictions already
made in earlier runs — not independent evidence.

This script fixes that with two changes:
  1. A completely disjoint slice of the dataset (rows 200-379) that no
     prior run in this study ever touched, so every prediction here is
     genuinely new.
  2. Full per-item logging (not just aggregate letter counts), so the
     result is auditable after the fact — you can trace any prediction
     back to its exact row, seed, and raw model output.

It also runs TWO statistical tests, not one:
  - A binomial test (treats every prediction as one independent trial)
  - A paired per-item test (treats every ITEM as one independent trial,
    averaged over its several exemplar-draw outcomes) — this is the
    more conservative, harder-to-challenge test, since it doesn't
    assume that showing the same question under different exemplars
    counts as fully independent evidence.
```

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install datasets transformers accelerate bitsandbytes sentencepiece pandas matplotlib tqdm scipy --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00


In [ ]:
import random
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from scipy.stats import binomtest, ttest_1samp, wilcoxon


In [ ]:
MODEL_NAME = "bigscience/mt0-xl"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)
print("Model parameter device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/797 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model parameter device: cuda:0


In [ ]:
# WHAT: rows 200-229 as the exemplar pool, rows 230-379 as the test
#       pool (150 items) — neither range overlaps rows 0-179, which
#       is everything touched by any prior run in this study.
def load_and_split_clean(lang_code: str):
    ds = load_dataset("facebook/belebele", lang_code, split="test")
    ds = ds.shuffle(seed=42)  # same canonical shuffle as before, for traceability
    exemplar_pool = ds.select(range(200, 230))   # untouched by any prior work
    test_pool = ds.select(range(230, 380))       # untouched by any prior work
    return exemplar_pool, test_pool


In [ ]:
LETTERS = ["A", "B", "C", "D"]

def format_question_block(row, show_answer: bool):
    options = [row["mc_answer1"], row["mc_answer2"], row["mc_answer3"], row["mc_answer4"]]
    block = f"Passage: {row['flores_passage']}\n"
    block += f"Question: {row['question']}\n"
    for letter, opt in zip(LETTERS, options):
        block += f"{letter}. {opt}\n"
    if show_answer:
        correct_letter = LETTERS[int(row["correct_answer_num"]) - 1]
        block += f"Answer: {correct_letter}\n"
    else:
        block += "Answer:"
    return block

def build_prompt(exemplars, target_row):
    parts = [format_question_block(ex, show_answer=True) for ex in exemplars]
    parts.append(format_question_block(target_row, show_answer=False))
    instructions = (
        "Answer each multiple-choice reading comprehension question "
        "using only the passage provided. Respond with a single "
        "letter (A, B, C, or D) and nothing else."
    )
    return instructions + "\n\n" + "\n\n".join(parts)


In [ ]:
def sample_exemplars(exemplar_pool, k, seed):
    rng = random.Random(seed)
    if k == 0:
        return []
    indices = rng.sample(range(len(exemplar_pool)), k)
    return [exemplar_pool[i] for i in indices]


In [ ]:
tokenizer.truncation_side = "left"

def call_llm(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=5120).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def parse_prediction(raw_output: str):
    for letter in LETTERS:
        if letter in raw_output.upper():
            return letter
    return None


In [ ]:
exemplar_pool, test_pool = load_and_split_clean("sin_Sinh")
exemplars = sample_exemplars(exemplar_pool, 8, 42)
prompt = build_prompt(exemplars, test_pool[0])

start = time.time()
_ = call_llm(prompt)
elapsed = time.time() - start

TOTAL_CALLS = 2 * 5 * 150  # languages x seeds x items
print(f"{elapsed:.2f}s per call")
print(f"Estimated total ({TOTAL_CALLS} calls): {elapsed * TOTAL_CALLS / 60:.1f} minutes")


README.md:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

sin_Sinh.jsonl:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

4.08s per call
Estimated total (1500 calls): 101.9 minutes


In [ ]:
# WHAT: 5 independent exemplar-draw seeds, on ONE fixed, fresh
#       150-item test set. Every prediction is logged with its
#       row index, so the result is fully auditable afterward.
CONFIG_CLEAN = {
    "languages": {"English": "eng_Latn", "Sinhala": "sin_Sinh"},
    "seeds": [42, 43, 44, 45, 46],
    "shot_count": 8,
    "output_dir": "/content/drive/MyDrive/belebele_results/results_clean",
}
os.makedirs(CONFIG_CLEAN["output_dir"], exist_ok=True)

per_item_records = []

for lang_name, lang_code in CONFIG_CLEAN["languages"].items():
    exemplar_pool, test_pool = load_and_split_clean(lang_code)

    for seed in CONFIG_CLEAN["seeds"]:
        exemplars = sample_exemplars(exemplar_pool, CONFIG_CLEAN["shot_count"], seed)

        for row_idx, row in enumerate(tqdm(test_pool, desc=f"{lang_name} | seed={seed} | CLEAN")):
            prompt = build_prompt(exemplars, row)
            try:
                raw = call_llm(prompt)
            except Exception as e:
                print(f"Error on row {row_idx}: {type(e).__name__}: {e}")
                torch.cuda.empty_cache()
                raw, pred = None, None
            else:
                pred = parse_prediction(raw)

            gold = LETTERS[int(row["correct_answer_num"]) - 1]
            per_item_records.append({
                "language": lang_name,
                "seed": seed,
                "row_idx": row_idx,
                "raw_output": raw,
                "parsed_pred": pred,
                "gold": gold,
                "correct": pred == gold,
                "selected_C": pred == "C",
            })

        torch.cuda.empty_cache()
        pd.DataFrame(per_item_records).to_csv(
            f"{CONFIG_CLEAN['output_dir']}/clean_per_item.csv", index=False
        )

clean_df = pd.DataFrame(per_item_records)
print(f"\nTotal predictions: {len(clean_df)}")
print(clean_df.groupby("language")["correct"].mean())


eng_Latn.jsonl:   0%|          | 0.00/836k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

English | seed=42 | CLEAN: 100%|██████████| 150/150 [03:53<00:00,  1.56s/it]
English | seed=43 | CLEAN: 100%|██████████| 150/150 [04:28<00:00,  1.79s/it]
English | seed=44 | CLEAN: 100%|██████████| 150/150 [04:22<00:00,  1.75s/it]
English | seed=45 | CLEAN: 100%|██████████| 150/150 [05:07<00:00,  2.05s/it]
English | seed=46 | CLEAN: 100%|██████████| 150/150 [04:21<00:00,  1.74s/it]
Sinhala | seed=42 | CLEAN: 100%|██████████| 150/150 [07:50<00:00,  3.14s/it]
Sinhala | seed=43 | CLEAN: 100%|██████████| 150/150 [09:13<00:00,  3.69s/it]
Sinhala | seed=44 | CLEAN: 100%|██████████| 150/150 [08:25<00:00,  3.37s/it]
Sinhala | seed=45 | CLEAN: 100%|██████████| 150/150 [11:01<00:00,  4.41s/it]
Sinhala | seed=46 | CLEAN: 100%|██████████| 150/150 [09:16<00:00,  3.71s/it]


Total predictions: 1500
language
English    0.608000
Sinhala    0.518667
Name: correct, dtype: float64


In [ ]:
# WHAT: treats every single prediction as one independent trial.
#       This is the easy headline number, now legitimately computed
#       on fully non-overlapping data.
print("\n=== TEST 1: Binomial test (per-prediction) ===")
for lang, group in clean_df.groupby("language"):
    n = len(group)
    c_count = int(group["selected_C"].sum())
    result = binomtest(c_count, n, p=0.25, alternative='greater')
    print(f"{lang}: C selected {c_count}/{n} = {c_count/n:.1%} | binomial p = {result.pvalue:.2e}")



=== TEST 1: Binomial test (per-prediction) ===
English: C selected 303/750 = 40.4% | binomial p = 1.67e-20
Sinhala: C selected 257/750 = 34.3% | binomial p = 9.23e-09


In [ ]:
# WHAT: treats every UNIQUE ITEM as one independent trial, first
#       averaging its outcome across the 5 exemplar-draw seeds.
#       This is the conservative version that pre-empts the
#       objection "the same question shown 5 times isn't 5
#       independent trials."
print("\n=== TEST 2: Per-item paired test (conservative) ===")
for lang, group in clean_df.groupby("language"):
    per_item_rate = group.groupby("row_idx")["selected_C"].mean()

    print(f"\n{lang}: {len(per_item_rate)} unique items, "
          f"each averaged over {group['seed'].nunique()} seeds")
    print(f"Mean per-item C-rate: {per_item_rate.mean():.1%} (chance = 25%)")

    t_result = ttest_1samp(per_item_rate, 0.25, alternative='greater')
    print(f"Paired t-test (item-level, n={len(per_item_rate)}): p = {t_result.pvalue:.2e}")

    w_result = wilcoxon(per_item_rate - 0.25, alternative='greater')
    print(f"Wilcoxon signed-rank (item-level): p = {w_result.pvalue:.2e}")



=== TEST 2: Per-item paired test (conservative) ===

English: 150 unique items, each averaged over 5 seeds
Mean per-item C-rate: 40.4% (chance = 25%)
Paired t-test (item-level, n=150): p = 2.52e-06
Wilcoxon signed-rank (item-level): p = 1.32e-03

Sinhala: 150 unique items, each averaged over 5 seeds
Mean per-item C-rate: 34.3% (chance = 25%)
Paired t-test (item-level, n=150): p = 1.47e-03
Wilcoxon signed-rank (item-level): p = 6.34e-02


In [ ]:
summary_rows = []
for lang, group in clean_df.groupby("language"):
    n = len(group)
    c_count = int(group["selected_C"].sum())
    binom = binomtest(c_count, n, p=0.25, alternative='greater')
    per_item_rate = group.groupby("row_idx")["selected_C"].mean()
    t_result = ttest_1samp(per_item_rate, 0.25, alternative='greater')
    w_result = wilcoxon(per_item_rate - 0.25, alternative='greater')
    summary_rows.append({
        "language": lang,
        "n_predictions": n,
        "n_items": len(per_item_rate),
        "n_seeds": group["seed"].nunique(),
        "C_rate_overall": c_count / n,
        "binomial_p": binom.pvalue,
        "paired_ttest_p": t_result.pvalue,
        "wilcoxon_p": w_result.pvalue,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{CONFIG_CLEAN['output_dir']}/clean_summary.csv", index=False)
print("\n=== FINAL SUMMARY ===")
print(summary_df.to_string(index=False))



=== FINAL SUMMARY ===
language  n_predictions  n_items  n_seeds  C_rate_overall   binomial_p  paired_ttest_p  wilcoxon_p
 English            750      150        5        0.404000 1.668716e-20        0.000003    0.001316
 Sinhala            750      150        5        0.342667 9.225555e-09        0.001469    0.063425
